# Decision Analysis 6 6

## Electre Tri-B

The objective of this laboratory session is to gain practical understanding of the **ELECTRE TRI-B** method – one of the most important *outranking-based sorting* methods used in Multi-Criteria Decision Analysis (MCDA).

The ELECTRE TRI-B method is designed to assign a finite set of decision alternatives to predefined, ordered categories based on their performance with respect to multiple, often conflicting criteria. Instead of producing a complete ranking, the method focuses on classification by comparing alternatives with reference profiles that define the boundaries between categories.

Within the method, each alternative is compared to a set of boundary profiles that separate adjacent categories. For each criterion, the difference between the performance of the alternative and the profile is evaluated. This comparison is then used to construct a **concordance index**, which expresses the degree to which there is sufficient evidence to support the assertion that the alternative is at least as good as the profile.

To account for strong opposition on individual criteria, **discordance indices** are also computed. These indices capture situations where a significant disadvantage on a single criterion may weaken or even invalidate the overall outranking relation.

The concordance and discordance information are aggregated into a **credibility index**, which represents the strength of the outranking relation between the alternative and the profile. This index is then compared with a predefined cutting level ($\lambda$) to determine whether the outranking relation is validated.

The assignment procedure is performed using one of two possible rules:

- **pessimistic (conjunctive) assignment** – the alternative is assigned to the highest category for which it sufficiently outranks the lower boundary profile,
- **optimistic (disjunctive) assignment** – the alternative is assigned to the lowest category whose upper boundary profile does not sufficiently outrank the alternative.

In the ELECTRE TRI-B method, the decision maker provides preference information in the following form:

- **q** – indifference threshold,
- **p** – preference threshold,
- **v** – veto threshold,
- **w** – weight of criterion *k*,
- **$\lambda$** – cutting level for validating the outranking relation,
- direction of preference (cost or benefit criterion).

During the laboratory sessions, we will consider a simplified version of the ELECTRE method, in which a single set of preference information is defined instead of a separate set for each profile.

> **IMPORTANT**
>
> Code written in this notebook will be checked against automatic code checker and the points will be given based on its' results, please leave the function signatures unchanged.
> As a result there are no partial points for a programing tasks

In [122]:
from pathlib import Path

import numpy as np
import pandas as pd

import utils

In [123]:
# Loading dataset and preference information
data_path = Path("")
dataset = utils.load_dataset(data_path)
boundary_profiles = utils.load_boundary_profiles(data_path)
preference_information = utils.load_preference_information(data_path)
credibility_threshold = 0.65

In [124]:
dataset

,g1,g2,g3,g4
Alternative,,,,
a1,90,86,46,30
a2,40,90,14,48
a3,94,100,40,36
a4,78,76,30,50
a5,60,60,30,30
a6,64,72,12,46
a7,62,88,22,48
a8,70,30,12,12


In [125]:
boundary_profiles

,g1,g2,g3,g4
Alternative,,,,
b1,64,61,32,32
b2,86,84,43,43


In [126]:
preference_information

,q,p,v,w,type
Criterion,,,,,
g1,2,6,20.0,40,gain
g2,2,5,24.0,30,gain
g3,0,2,NaN,25,gain
g4,0,2,NaN,5,gain


### Task 1 (maximum points:  1)


Implement the `difference_function` function.

$$
    d_j (a, b) = \left\{ \begin{array}{ll}
    g_j(a) - g_j(b), & \textrm{for \textit{gain} criterion},\\
    g_j(b) - g_j(a), & \textrm{for \textit{cost} criterion},\\
    \end{array} \right.
$$

This difference is used later to compute marginal concordance and discordance values.

In [127]:
def difference_function(alternative_a: float, alternative_b: float, criterion_type: utils.CriterionType) -> float:
    """
    Function that calculates the difference between given alternative pair on a single criterion.

    :param alternative_a: first alternative in a pair
    :param alternative_b: second alternative in a pair
    :param criterion_type: criterion type either gain or cost
    :return: difference between alternative pair calculated according to the criterion type
    """
    if criterion_type == utils.CriterionType.GAIN:
        return alternative_a - alternative_b
    else:
        return alternative_b - alternative_a
    

### Task 2 (maximum points:  1)


Implement the `calculate_marginal_concordance_index` function.

Requirements:
- Use inputs `diff`, `indifference_threshold` ($q$), and `preference_threshold` ($p$).
- Compute the marginal concordance value with the piecewise rule:

$$
c(a, b)=\begin{cases}
1, & d(a,b) \ge -q \\
0, & d(a,b) \le -p \\
\frac{p + d(a,b)}{p - q}, & -q > d(a,b) > -p
\end{cases}
$$

In [128]:
def calculate_marginal_concordance_index(diff: float, indifference_threshold: float, preference_threshold: float) -> float:
    """
    Function that calculates the marginal concordance index for the given pair of alternatives, according to the formula presented during classes.

    :param diff: difference between compared alternatives either as a float for single criterion and alternative pairs, or as numpy array for multiple alternatives
    :param indifference_threshold: indifference threshold either as a float if you prefer to calculate for a single criterion or as numpy array for multiple criterion
    :param preference_threshold: preference threshold either as a float if you prefer to calculate for a single criterion or as numpy array for multiple criterion
    :return: marginal concordance index either as a float for single criterion and alternative pairs, or as numpy array for multiple criterion
    """
    if diff >= -indifference_threshold:
        return 1.0
    elif diff <= -preference_threshold:
        return 0.0
    else:
        return (preference_threshold+diff)/(preference_threshold-indifference_threshold)

### Task 3 (maximum points:  1)


Implement the `calculate_marginal_concordance_matrix` function.

Requirements:
- Use `dataset`, `boundary_profiles`, and `preference_information`.
- Build differences for both directions: alternative vs profile and profile vs alternative.
- Use `q` and `p` thresholds from `preference_information`.
- Return a 4D matrix with shape `[2, n_alternatives, n_profiles, n_criteria]`.

In [129]:
def calculate_marginal_concordance_matrix(
    dataset: pd.DataFrame,
    boundary_profiles: pd.DataFrame,
    preference_information: pd.DataFrame
) -> np.ndarray:
    """
    Function that calculates the marginal concordance matrix for all alternatives pairs and criterion available in dataset

    :param dataset: pandas dataframe representing dataset with alternatives as rows and criterion as columns
    :param boundary_profiles: pandas dataframe with boundary profiles
    :param preference_information: pandas dataframe with preference information for all criterion
    :return: 4D numpy array with marginal concordance matrix with shape [2, number of alternatives, number of boundary profiles, number of criterion], where element with index [0, i, j, k] describe marginal concordance index between alternative i and boundary profile j on criterion k, while element with index [1, i, j, k] describe marginal concordance index between boundary profile j and alternative i on criterion k
    """
    n_alts = len(dataset)
    n_profiles = len(boundary_profiles)
    n_criteria = len(dataset.columns)

    result = np.zeros((2, n_alts, n_profiles, n_criteria))

    q = preference_information['q'].values
    p = preference_information['p'].values
    types = preference_information['type'].values

    for i in range(n_alts):
        for j in range(n_profiles):
            for k in range(n_criteria):
                a_val = dataset.iloc[i, k]
                b_val = boundary_profiles.iloc[j, k]
                ctype = types[k]

                # alt vs profile
                diff_ab = difference_function(a_val, b_val, ctype)
                result[0, i, j, k] = calculate_marginal_concordance_index(diff_ab, q[k], p[k])

                # profile vs alt
                diff_ba = difference_function(b_val, a_val, ctype)
                result[1, i, j, k] = calculate_marginal_concordance_index(diff_ba, q[k], p[k])

    return result

In [130]:
marginal_concordance_matrix = calculate_marginal_concordance_matrix(dataset, boundary_profiles, preference_information)

In [131]:
marginal_concordance_matrix[:, 4, 0]

array([[0.5, 1. , 0. , 0. ],
       [1. , 1. , 1. , 1. ]])

### Task 4 (maximum points:  1)


Implement the `calculate_comprehensive_concordance_matrix` function.

The function should aggregate marginal concordance values across all criteria using criterion weights.

For each pair of alternatives and boundary profile (a, b), compute the comprehensive concordance index:

$$C(a, b) = \frac{\sum w_k * c_k(a, b)}{\sum w_k}$$

where:
- $w_k$ is the weight of criterion k,
- $c_k(a, b)$ is the marginal concordance value.


In [ ]:
def calculate_comprehensive_concordance_matrix(marginal_concordance_matrix: np.ndarray, preference_information: pd.DataFrame) -> np.ndarray:
    """
    Function that calculates comprehensive concordance matrix for the given dataset

    :param marginal_concordance_matrix: 4D numpy array with marginal concordance matrix with shape [2, number of alternatives, number of boundary profiles, number of criterion], where element with index [0, i, j, k] describe marginal concordance index between alternative i and boundary profile j on criterion k, while element with index [1, i, j, k] describe marginal concordance index between boundary profile j and  alternative i on criterion k
    :param preference_information: pandas dataframe containing preference information for all criterion, including their weights in column 'w'
    :return: 3D numpy array with comprehensive concordance matrix with shape [2, number of alternatives, number of boundary profiles], where element with index [0, i, j] describe comprehensive concordance index between alternative i and boundary profile j, while element with index [1, i, j] describe comprehensive concordance index between boundary profile j and  alternative i
    """
    weights = preference_information['w'].values
    comprehensive = np.sum(marginal_concordance_matrix * weights, axis=3) / weights.sum()
    return comprehensive

In [133]:
comprehensive_concordance_index = calculate_comprehensive_concordance_matrix(marginal_concordance_matrix, preference_information)

In [134]:
comprehensive_concordance_index

array([[[0.95, 0.95],
        [0.35, 0.35],
        [1.  , 0.7 ],
        [0.75, 0.05],
        [0.5 , 0.  ],
        [0.75, 0.05],
        [0.75, 0.35],
        [0.4 , 0.  ]],

       [[0.05, 0.55],
        [0.65, 0.65],
        [0.  , 0.3 ],
        [0.25, 0.95],
        [1.  , 1.  ],
        [0.65, 0.95],
        [0.65, 0.75],
        [0.6 , 1.  ]]])

### Task 5 (maximum points:  1)


Implement the `calculate_marginal_discordance_index` function.

Requirements:
- Use inputs `diff`, `ppreference_threshold` (p), and `veto threshold` (v).
- Compute the marginal discordance value with the piecewise rule:

$$
    D_j (a_i, b_{h}) = \left\{ \begin{array}{ll}
        1, & \textrm{if } d_k(a_{i}, b_{h}) \leq -v_j^h),\\
        0, & \textrm{if } d_k(a_{i}, b_{h}) \geq -p_j^h),\\
        \frac{-d_k(a_{i}, b_{h}) - p_j^h}{v_j^h - p_j^h}, & \textrm{if } -p_j^h > d_k(a_{i}, b_{h}) > -v_j^h).
    \end{array} \right.
$$

In [135]:
def calculate_marginal_discordance_index(diff: float, preference_threshold: float, veto_threshold: float) -> float:
    """
    Function that calculates the marginal discordance index for the given pair of alternatives, according to the formula presented during classes.

    :param diff: difference between compared alternatives either as a float for single criterion and alternative pairs, or as numpy array for multiple alternatives
    :param preference_threshold: preference threshold either as a float if you prefer to calculate for a single criterion or as numpy array for multiple criterion
    :param veto_threshold: veto threshold either as a float if you prefer to calculate for a single criterion or as numpy array for multiple criterion
    :return: marginal discordance index either as a float for single criterion and alternative pairs, or as numpy array for multiple criterion
    """
    if diff <= -veto_threshold:
        return 1.0
    elif diff >= -preference_threshold:
        return 0.0
    else:
        return (-diff - preference_threshold) / (veto_threshold - preference_threshold)

### Task 6 (maximum points:  1)


Implement the `calculate_marginal_discordance_matrix` function.

Requirements:
- Use `dataset`, `boundary_profiles`, `preference_thresholds`, `veto_thresholds`, and `criterion_types`.
- Build differences for both directions: alternative vs profile and profile vs alternative.
- Compute discordance values using `calculate_marginal_discordance_index`.
- Return a 4D matrix with shape `[2, n_alternatives, n_profiles, n_criteria]`.

In [136]:
def calculate_marginal_discordance_matrix(dataset: pd.DataFrame, boundary_profiles: pd.DataFrame, preference_thresholds, veto_thresholds, criterion_types) -> np.ndarray:
    """
    Function that calculates the marginal discordance matrix for all alternatives pairs and criterion available in dataset

    :param dataset: pandas dataframe representing dataset with alternatives as rows and criterion as columns
    :param boundary_profiles: pandas dataframe with boundary profiles
    :param preference_thresholds: pandas dataframe representing preference thresholds for all boundary profiles and criterion
    :param veto_thresholds: pandas dataframe representing veto thresholds for all boundary profiles and criterion
    :param criterion_types: pandas dataframe with a column 'type' representing the type of criterion (either gain or cost)
    :return: 4D numpy array with marginal discordance matrix with shape [2, number of alternatives, number of boundary profiles, number of criterion], where element with index [0, i, j, k] describe marginal discordance index between alternative i and boundary profile j on criterion k, while element with index [1, i, j, k] describe marginal discordance index between boundary profile j and  alternative i on criterion k
    """
    n_alts = len(dataset)
    n_profiles = len(boundary_profiles)
    n_criteria = len(dataset.columns)

    result = np.zeros((2, n_alts, n_profiles, n_criteria))

    p = preference_thresholds['p'].values
    v = veto_thresholds['v'].values
    types = criterion_types['type'].values

    for i in range(n_alts):
        for j in range(n_profiles):
            for k in range(n_criteria):
                a_val = dataset.iloc[i, k]
                b_val = boundary_profiles.iloc[j, k]
                ctype = types[k]

                if pd.isna(v[k]):
                    continue

                diff_ab = difference_function(a_val, b_val, ctype)
                result[0, i, j, k] = calculate_marginal_discordance_index(diff_ab, p[k], v[k])

                diff_ba = difference_function(b_val, a_val, ctype)
                result[1, i, j, k] = calculate_marginal_discordance_index(diff_ba, p[k], v[k])

    return result

### Task 7 (maximum points:  1)


Implement the `calculate_credibility_index` function.

The function should compute the credibility index by combining the comprehensive concordance matrix with marginal discordance indices.

For each pair of alternatives and boundary profile (a, b), compute the credibility index:

$$\sigma(a, b) = C(a, b) \cdot \prod_{d_k(a, b) > C(a, b)} \frac{1 - d_k(a, b)}{1 - C(a, b)}$$

where:
- $C(a, b)$ is the comprehensive concordance index,
- $d_k(a, b)$ is the marginal discordance index on criterion k,
- the product is taken over criteria where discordance exceeds concordance.

In [ ]:
def calculate_credibility_index(comprehensive_concordance_matrix: np.ndarray, marginal_discordance_matrix: np.ndarray) -> np.ndarray:
    """
    Function that calculates the credibility index for the given comprehensive concordance matrix and marginal discordance matrix

    :param comprehensive_concordance_matrix: 3D numpy array with comprehensive concordance matrix. Every entry in the matrix [i, j] represents comprehensive concordance index between alternative i and alternative j
    :param marginal_discordance_matrix: 3D numpy array with marginal discordance matrix, Consecutive indices [i, j, k] describe first alternative, second alternative, criterion
    :return: 3D numpy array with credibility matrix with shape [2, number of alternatives, number of boundary profiles], where element with index [0, i, j] describe credibility index between alternative i and boundary profile j, while element with index [1, i, j] describe credibility index between boundary profile j and  alternative i
    """
    C = comprehensive_concordance_matrix
    D = marginal_discordance_matrix
    credibility = np.zeros_like(C)
    for direction in range(2):
        for i in range(C.shape[1]):
            for j in range(C.shape[2]):
                c = C[direction, i, j]
                product = 1.0
                for k in range(D.shape[3]):
                    d = D[direction, i, j, k]
                    if d > c:
                        product *= (1.0 - d) / (1.0 - c)
                credibility[direction, i, j] = c * product
    return credibility

### Task 8 (maximum points:  1)


Implement the `calculate_outranking_relation_matrix` function.

The function should compare credibility values with the cutting level $\lambda$ (`credibility_threshold`) and return a boolean matrix indicating whether outranking holds for each pair.

In [138]:
def calculate_outranking_relation_matrix(credibility_index: np.ndarray, credibility_threshold: float) -> np.ndarray:
    """
    Function that calculates boolean matrix with information if outranking holds for a given pair

    :param credibility_index: 3D numpy array with credibility matrix with shape [2, number of alternatives, number of boundary profiles], where element with index [0, i, j] describe credibility index between alternative i and boundary profile j, while element with index [1, i, j] describe credibility index between boundary profile j and  alternative i
    :param credibility_threshold: float number
    :return: 3D numpy boolean matrix with information if outranking holds for a given pair
    """
    return credibility_index >= credibility_threshold

### Task 9 (maximum points:  1)


Implement the `calculate_pessimistic_assigment` function.

Use the pessimistic rule: assign each alternative to the highest category for which the alternative is at least as good as the boundary profile.

In [ ]:
def calculate_pessimistic_assigment(relation: pd.DataFrame) -> pd.DataFrame:
    """
    Function that calculates pessimistic assigment for given relation between alternatives and boundary profiles

    :param relation: pandas dataframe with relation between alternatives as rows and boundary profiles as columns. With "<" or ">" for preference, "I" for indifference and "?" for incompatibility
    :return: dataframe with pessimistic assigment
    """
    profiles = relation.columns.tolist()
    assignments = {}
    
    for alt in relation.index:
        assigned_category = 1

        for h in range(len(profiles) - 1, -1, -1):
            prof = profiles[h]
            rel = relation.loc[alt, prof]

            # alt outranks profile if relation is '>' or 'I'
            if rel in ['>', 'I']:
                assigned_category = h + 2
                break
        assignments[alt] = assigned_category

    result = pd.DataFrame(list(assignments.values()),
                      index=list(assignments.keys()),
                      columns=['pessimistic_assignment'])
    return result

### Task 10 (maximum points:  1)


Implement the `calculate_optimistic_assigment` function.

Use the optimistic rule: assign each alternative to the lowest category whose upper boundary profile does not sufficiently outrank the alternative.

In [ ]:
def calculate_optimistic_assigment(relation: pd.DataFrame) -> pd.DataFrame:
    """
   Function that calculates optimistic assigment for given relation between alternatives and boundary profiles

   :param relation: pandas dataframe with relation between alternatives as rows and boundary profiles as columns. With "<" or ">" for preference, "I" for indifference and "?" for incompatibility
   :return: dataframe with optimistic assigment
   """
    profiles = relation.columns.tolist()
    n_categories = len(profiles) + 1

    assignments = {}
    for alt in relation.index:
        assigned_category = n_categories

        for h in range(len(profiles)):
            prof = profiles[h]
            rel = relation.loc[alt, prof]
            # profile outranks alt if relation is '<' or 'I'
            if rel in ['<', 'I']:
                assigned_category = h + 1
                break
        assignments[alt] = assigned_category

    result = pd.DataFrame(list(assignments.values()),
                      index=list(assignments.keys()),
                      columns=['optimistic_assignment'])
    return result

## HOMEWORK

**Deadline:** 29.04.2026, 23:59

Using your implementations of the **PROMETHEE** and **ELECTRE** methods prepared during the laboratory sessions, perform an analysis of the dataset provided for the UTA laboratories.

As a solution, submit:
- Jupyter notebooks for:
  - PROMETHEE (max 7 points)
  - ELECTRE (max 10 points)
- A report in **PDF format**, in which you answer the questions listed below.

### PROMETHEE (max points: 10)

- Describe the preference information used as input to the method. *(2.0 points)*  
- Present the final results obtained using the method. *(2.0 points)*  
- Compare the complete and partial rankings. *(2.0 points)*  
- Discuss your findings regarding the method, including:
  - the performance of the best and worst alternatives,  
  - whether the pairwise comparisons defined in the dataset report are satisfied. *(4.0 points)*  

### ELECTRE (max points: 10)

- Describe the preference information used as input to the method. *(2.0 points)*  
- Present the final results obtained using the method. *(2.0 points)*  
- Compare the optimistic and pessimistic class assignments. *(2.0 points)*  
- Discuss your findings regarding the method, including:
  - the performance of the best and worst alternatives,  
  - whether the pairwise comparisons defined in the dataset report are satisfied. *(4.0 points)*  

### Method Comparison (max points: 3)

- Comment on the similarities and differences between the methods. *(3.0 points)*  

**Final grade:**  
The final grade will be based on the total number of points obtained.